In [1]:
import pandas as pd
import numpy as np
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import DatasetDriftMetric, ColumnDriftMetric
import warnings
warnings.filterwarnings('ignore')

print("Evidently imported successfully")
print(f"Evidently version: {pd.__version__}")

Evidently imported successfully
Evidently version: 2.3.3


In [2]:
# Load processed data
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

# Reference data = training data (what model was trained on)
reference_data = X_train.copy()
reference_data['Machine failure'] = y_train.values

# Current data = test data (simulating new production data)
current_data = X_test.copy()
current_data['Machine failure'] = y_test.values

print(f"Reference data shape: {reference_data.shape}")
print(f"Current data shape:   {current_data.shape}")

Reference data shape: (8000, 9)
Current data shape:   (2000, 9)


In [3]:
# Simulate a drifted production dataset (as if sensors degraded over time)
np.random.seed(42)
drifted_data = X_test.copy()

# Introduce drift: shift temperature and torque distributions
drifted_data['Air temperature [K]'] += np.random.normal(2.5, 1.0, len(drifted_data))
drifted_data['Process temperature [K]'] += np.random.normal(3.0, 1.0, len(drifted_data))
drifted_data['Torque [Nm]'] += np.random.normal(8.0, 2.0, len(drifted_data))
drifted_data['Tool wear [min]'] += np.random.normal(30.0, 5.0, len(drifted_data))
drifted_data['Machine failure'] = y_test.values

print("Drifted dataset created")
print(f"Original Torque mean:  {X_test['Torque [Nm]'].mean():.2f}")
print(f"Drifted Torque mean:   {drifted_data['Torque [Nm]'].mean():.2f}")

Drifted dataset created
Original Torque mean:  39.92
Drifted Torque mean:   47.83


In [4]:
# Report 1: No drift expected (reference vs current/test)
report_no_drift = Report(metrics=[DataDriftPreset()])
report_no_drift.run(reference_data=reference_data, current_data=current_data)
report_no_drift.save_html('data/report_no_drift.html')
print("No-drift report saved to data/report_no_drift.html")

No-drift report saved to data/report_no_drift.html


In [5]:
# Report 2: Drift expected (reference vs drifted production data)
report_drift = Report(metrics=[DataDriftPreset()])
report_drift.run(reference_data=reference_data, current_data=drifted_data)
report_drift.save_html('data/report_drift_detected.html')
print("Drift report saved to data/report_drift_detected.html")

Drift report saved to data/report_drift_detected.html


In [6]:
# Extract drift summary programmatically
result_no_drift = report_no_drift.as_dict()
result_drift = report_drift.as_dict()

# Get dataset-level drift result
no_drift_detected = result_no_drift['metrics'][0]['result']['dataset_drift']
drift_detected = result_drift['metrics'][0]['result']['dataset_drift']

print("=== Drift Detection Summary ===")
print(f"Reference vs Test data  -> Dataset drift detected: {no_drift_detected}")
print(f"Reference vs Drifted data -> Dataset drift detected: {drift_detected}")

=== Drift Detection Summary ===
Reference vs Test data  -> Dataset drift detected: False
Reference vs Drifted data -> Dataset drift detected: False
